# Mask2Former (Swin-S) - Uydudan bina hasari segmentasyonu

**Katman:** Guclendirici

M1'de SegFormer'i geciyor (mIoU %76,1 vs %74,4). smp'de yok - HF transformers ile.  
Ayni veri, ayni metrik fonksiyonu (`seg_common.evaluate_seg` uyumlu cikti).

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/02_bina_hasari
    %pip install -q transformers roboflow albumentations opencv-python-headless
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import seg_common as SC
SC.CONFIG.update(dict(
    source='roboflow',
    roboflow_api_key='BURAYA_ROBOFLOW_API_KEY',
    roboflow_workspace='workspace-slug',
    roboflow_project='project-slug',
    roboflow_version=1,
    num_classes=1,   # xBD 4 hasar sinifi kullanilacaksa 4
))
paths = SC.prepare_dataset(); dl = SC.build_loaders(paths)
print({k: len(v.dataset) for k, v in dl.items()})

## 2. Model

In [ ]:
MODEL_NAME = "mask2former_swin_s"
HF_CKPT = "facebook/mask2former-swin-small-ade-semantic"
PARAMS = dict(ckpt=HF_CKPT, epochs=SC.CONFIG['epochs'], lr=1e-5)
import torch
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor
proc = Mask2FormerImageProcessor(reduce_labels=False, ignore_index=255,
                                 do_resize=True, size={'height': SC.CONFIG['img_size'], 'width': SC.CONFIG['img_size']})
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    HF_CKPT, num_labels=2, ignore_mismatched_sizes=True)

## 3. Egitim dongusu (basit)

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; model.to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=PARAMS['lr'])
for ep in range(PARAMS['epochs']):
    model.train(); tot = 0
    for img, m in dl['train']:
        # img: (B,3,H,W) 0..1 normalize edilmis;  m: (B,1,H,W) 0/1
        inp = proc(images=list((img*255).byte().permute(0,2,3,1).numpy()),
                   segmentation_maps=list(m.squeeze(1).long().numpy()), return_tensors='pt')
        inp = {k: v.to(dev) for k, v in inp.items()}
        out = model(**inp); out.loss.backward(); opt.step(); opt.zero_grad(); tot += out.loss.item()
    print(f'epoch {ep+1} loss {tot/len(dl["train"]):.4f}')

## 4. Test (seg maskesine cevir) + metrik

In [ ]:
import numpy as np
def m2f_prob(model, loader):
    class Wrap:  # evaluate_seg imzasina uydur: sigmoid(logits) bekliyor
        pass
    tp=fp=fn=tn=0
    model.eval()
    with torch.no_grad():
        for img, m in loader:
            inp = proc(images=list((img*255).byte().permute(0,2,3,1).numpy()), return_tensors='pt')
            inp = {k: v.to(dev) for k, v in inp.items()}
            seg = proc.post_process_semantic_segmentation(model(**inp),
                     target_sizes=[img.shape[-2:]]*img.shape[0])
            for s, gt in zip(seg, m):
                pred = (s.cpu().numpy() == 1).astype(int).ravel()
                g = gt.numpy().astype(int).ravel()
                tp += int(((pred==1)&(g==1)).sum()); fp += int(((pred==1)&(g==0)).sum())
                fn += int(((pred==0)&(g==1)).sum()); tn += int(((pred==0)&(g==0)).sum())
    e=1e-9; iou=tp/(tp+fp+fn+e); iou_bg=tn/(tn+fp+fn+e)
    prec=tp/(tp+fp+e); rec=tp/(tp+fn+e); spec=tn/(tn+fp+e)
    f1=2*prec*rec/(prec+rec+e)
    return {'iou_fg':iou,'iou_bg':iou_bg,'mean_iou':(iou+iou_bg)/2,'f1':f1,'precision':prec,
            'recall':rec,'specificity':spec,'accuracy':(tp+tn)/(tp+tn+fp+fn+e),'auc':float('nan')}
split = 'test' if 'test' in dl else 'valid'
metrics = m2f_prob(model, dl[split])
eff = {'params_m': sum(p.numel() for p in model.parameters())/1e6, 'gflops': float('nan')}
lb = SC.save_result(MODEL_NAME, metrics, eff, extra={'params': PARAMS}); print(metrics); lb